# RO-069 Driving to a point, a line and a pose: every number in the Note

The robot, its wheel limit and the three controllers live in `images/drivekit.py`: wheel separation 0.2 m, each wheel at
most 0.6 m/s, commands at 20 Hz held for 0.05 s, Euler steps of 0.01 s. The robot starts at (2.0, 1.0) facing 30 deg
and must stop at the dock pose (3.6, 3.5) facing 90 deg. Nothing here is random.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import drivekit as d
deg = np.degrees

## 1. The opening numbers

In [2]:
rho, a, b = d.polar(d.START)
print(f'distance rho = {rho:.4f} m, line of sight {deg(a + d.START[2]):.2f} deg')
print(f'alpha = {deg(a):.2f} deg, beta = {deg(b):.2f} deg, heading still to turn = {90 - 30} deg')

distance rho = 2.9682 m, line of sight 57.38 deg
alpha = 27.38 deg, beta = 32.62 deg, heading still to turn = 60 deg


## 2. Driving to a point: v = 1 x rho, w = 5 x alpha

In [3]:
v, w = d.point_law(d.START)
print(f'raw command: v = {v:.4f} m/s, w = {w:.4f} rad/s; wheels {v + 0.1 * w:.4f} and {v - 0.1 * w:.4f} m/s')
vl, wl = d.limit(v, w)
print(f'scale = 0.6 / {v + 0.1 * w:.4f} = {0.6 / (v + 0.1 * w):.4f}')
print(f'scaled: v = {vl:.4f} m/s, w = {wl:.4f} rad/s; wheels {vl + 0.1 * wl:.4f} and {vl - 0.1 * wl:.4f} m/s')
print(f'turning radius before {v / w:.4f} m, after {vl / wl:.4f} m')
P = d.run(d.point_law)
POL = np.array([d.polar(q) for q in zip(P['x'], P['y'], P['th'])])
for t in range(9):
    k = 100 * t
    print(f't = {t} s: rho {POL[k, 0]:.4f} m, alpha {deg(POL[k, 1]):.2f} deg, heading {deg(P["th"][k]):.2f} deg,'
          f' v {P["v"][k]:.4f}, w {P["w"][k]:.4f}')
k6 = int(np.argmax(POL[:, 0] <= 0.6))
print(f'rho reaches 0.6 m at {P["t"][k6]:.2f} s; one second later {POL[k6 + 100, 0]:.4f} m = {POL[k6 + 100, 0] / POL[k6, 0]:.3f} of it')
print(f'arrives (within 1 cm) at {P["t"][-1]:.2f} s at ({P["x"][-1]:.3f}, {P["y"][-1]:.3f}) facing {deg(P["th"][-1]):.2f} deg,'
      f' {90 - deg(P["th"][-1]):.2f} deg short')

raw command: v = 2.9682 m/s, w = 2.3894 rad/s; wheels 3.2071 and 2.7292 m/s
scale = 0.6 / 3.2071 = 0.1871
scaled: v = 0.5553 m/s, w = 0.4470 rad/s; wheels 0.6000 and 0.5106 m/s
turning radius before 1.2422 m, after 1.2422 m
t = 0 s: rho 2.9682 m, alpha 27.38 deg, heading 30.00 deg, v 0.5553, w 0.4470
t = 1 s: rho 2.4340 m, alpha 11.59 deg, heading 49.58 deg, v 0.5761, w 0.2394
t = 2 s: rho 1.8560 m, alpha 3.78 deg, heading 59.31 deg, v 0.5895, w 0.1047
t = 3 s: rho 1.2630 m, alpha 0.77 deg, heading 63.06 deg, v 0.5968, w 0.0316
t = 4 s: rho 0.6645 m, alpha 0.05 deg, heading 63.95 deg, v 0.5996, w 0.0039
t = 5 s: rho 0.2402 m, alpha 0.00 deg, heading 64.01 deg, v 0.2402, w 0.0000
t = 6 s: rho 0.0861 m, alpha 0.00 deg, heading 64.01 deg, v 0.0861, w 0.0000
t = 7 s: rho 0.0309 m, alpha 0.00 deg, heading 64.01 deg, v 0.0309, w 0.0000
t = 8 s: rho 0.0111 m, alpha 0.00 deg, heading 64.01 deg, v 0.0111, w 0.0000
rho reaches 0.6 m at 4.11 s; one second later 0.2147 m = 0.359 of it
arrives (wit

Why alpha shrinks near the goal, at easy numbers (v = 1 x rho): the rate of alpha is v sin(alpha) / rho - 5 alpha, about (1 - 5) alpha.

In [4]:
for al in (0.1, 0.05):
    r = 0.5
    rate = 1 * r * np.sin(al) / r - 5 * al
    print(f'alpha = {al}: rate {rate:.4f} rad/s, -4 alpha = {-4 * al:.4f}')
P3 = d.run(lambda q: d.point_law(q, k_alpha=3))
print(f'with k_alpha = 3 the robot arrives facing {deg(P3["th"][-1]):.2f} deg')

alpha = 0.1: rate -0.4002 rad/s, -4 alpha = -0.4000
alpha = 0.05: rate -0.2000 rad/s, -4 alpha = -0.2000
with k_alpha = 3 the robot arrives facing 70.74 deg


## 3. Driving along a line: v = 0.5 m/s, w = -2 e - 2 psi

In [5]:
e, psi = d.line_dist(d.START)
v, w = d.line_law(d.START)
print(f'e = {e:.4f} m, psi = {deg(psi):.2f} deg = {psi:.4f} rad; raw w = {w:.4f} rad/s')
vl, wl = d.limit(v, w)
print(f'wheels {v + 0.1 * w:.4f}, {v - 0.1 * w:.5f} -> scale 0.6 / {v - 0.1 * w:.5f} = {0.6 / (v - 0.1 * w):.4f};'
      f' scaled v {vl:.4f}, w {wl:.5f}')
L = d.run(d.line_law, stop=lambda q: q[1] + d.R_ROBOT >= d.WALL_Y)
for t in range(7):
    k = 100 * t
    ee, pp = d.line_dist((L['x'][k], L['y'][k], L['th'][k]))
    print(f't = {t} s: e {ee:.4f} m, psi {deg(pp):.2f} deg, heading {deg(L["th"][k]):.2f}')
ee, pp = d.line_dist((L['x'][-1], L['y'][-1], L['th'][-1]))
print(f'front reaches the dock at {L["t"][-1]:.2f} s at {L["v"][-2]:.2f} m/s; e = {ee:.4f} m, heading {deg(L["th"][-1]):.2f} deg;'
      f' lowest heading {deg(L["th"].min()):.2f} deg')
spring, damper = 0.5 * 2, 2
print(f'spring v K_d = {spring}, damper K_h = {damper}, natural frequency {np.sqrt(spring):.3f} rad/s, damping ratio {damper / (2 * np.sqrt(spring)):.3f}')

e = 1.6000 m, psi = -60.00 deg = -1.0472 rad; raw w = -1.1056 rad/s
wheels 0.3894, 0.61056 -> scale 0.6 / 0.61056 = 0.9827; scaled v 0.4914, w -1.08648
t = 0 s: e 1.6000 m, psi -60.00 deg, heading 30.00
t = 1 s: e 1.1240 m, psi -72.63 deg, heading 17.37
t = 2 s: e 0.6836 m, psi -50.87 deg, heading 39.13
t = 3 s: e 0.3644 m, psi -29.75 deg, heading 60.25
t = 4 s: e 0.1769 m, psi -15.47 deg, heading 74.53
t = 5 s: e 0.0811 m, psi -7.43 deg, heading 82.57
t = 6 s: e 0.0359 m, psi -3.39 deg, heading 86.61
front reaches the dock at 6.60 s at 0.50 m/s; e = 0.0218 m, heading 87.92 deg; lowest heading 13.70 deg
spring v K_d = 1.0, damper K_h = 2, natural frequency 1.000 rad/s, damping ratio 1.000


## 4. Driving to a pose: v = 1 x rho, w = 5 alpha - 4 beta

In [6]:
v, w = d.pose_law(d.START)
rho, a, b = d.polar(d.START)
print(f'5 alpha = {5 * a:.4f}, -4 beta = {-4 * b:.4f}, w = {w:.4f} rad/s, v = {v:.4f} m/s')
vl, wl = d.limit(v, w)
print(f'scale {0.6 / (v + 0.1 * w):.4f}: v {vl:.4f} m/s, w {wl:.4f} rad/s')
Q = d.run(d.pose_law)
QP = np.array([d.polar(q) for q in zip(Q['x'], Q['y'], Q['th'])])
for t in range(9):
    k = 100 * t
    print(f't = {t} s: rho {QP[k, 0]:.4f}, alpha {deg(QP[k, 1]):.2f}, beta {deg(QP[k, 2]):.2f}, heading {deg(Q["th"][k]):.2f},'
          f' v {Q["v"][k]:.4f}, w {Q["w"][k]:.4f}')
print(f'arrives at {Q["t"][-1]:.2f} s at ({Q["x"][-1]:.3f}, {Q["y"][-1]:.3f}) facing {deg(Q["th"][-1]):.3f} deg; largest x {Q["x"].max():.3f}')

5 alpha = 2.3894, -4 beta = -2.2773, w = 0.1122 rad/s, v = 2.9682 m/s
scale 0.2014: v 0.5977 m/s, w 0.0226 rad/s
t = 0 s: rho 2.9682, alpha 27.38, beta 32.62, heading 30.00, v 0.5977, w 0.0226
t = 1 s: rho 2.4481, alpha 28.40, beta 26.64, heading 34.95, v 0.5852, w 0.1479
t = 2 s: rho 1.9296, alpha 24.47, beta 19.77, heading 45.75, v 0.5774, w 0.2261
t = 3 s: rho 1.3937, alpha 17.45, beta 12.60, heading 59.95, v 0.5735, w 0.2647
t = 4 s: rho 0.8362, alpha 8.97, beta 5.86, heading 75.17, v 0.5743, w 0.2565
t = 5 s: rho 0.3300, alpha 2.14, beta 1.28, heading 86.58, v 0.3300, w 0.0977
t = 6 s: rho 0.1183, alpha 0.37, beta 0.21, heading 89.41, v 0.1183, w 0.0177
t = 7 s: rho 0.0424, alpha 0.06, beta 0.03, heading 89.91, v 0.0424, w 0.0029
t = 8 s: rho 0.0152, alpha 0.01, beta 0.01, heading 89.99, v 0.0152, w 0.0005
arrives at 8.45 s at (3.600, 3.490) facing 89.994 deg; largest x 3.600


The error equations near the dock: spring = -k_rho k_beta, damper = k_alpha - k_rho; the characteristic equation and its roots.

In [7]:
kr, ka, kb = d.K_RHO, d.K_ALPHA, d.K_BETA
spring, damper = -kr * kb, ka - kr
wn = np.sqrt(spring); zeta = damper / (2 * wn)
print(f'spring {spring}, damper {damper}, natural frequency {wn} rad/s, damping ratio {zeta}')
print('roots of s^2 + 4 s + 4:', np.roots([1, damper, spring]))
# check: the linear error equations give the same beta'' + 4 beta' + 4 beta = 0
A = np.array([[-(ka - kr), -kb], [-kr, 0.0]])
print('matrix of (alpha, beta):', A.tolist(), 'eigenvalues', np.linalg.eigvals(A))
# settling time of a critically damped system with natural frequency 2 rad/s (2 % band), from release with no speed
t = np.arange(0, 6, 1e-4); x = (1 + wn * t) * np.exp(-wn * t)
print(f'2 % settling time {t[np.where(np.abs(x) > 0.02)[0][-1]]:.2f} s')

spring 4.0, damper 4.0, natural frequency 2.0 rad/s, damping ratio 1.0
roots of s^2 + 4 s + 4: [-2. -2.]
matrix of (alpha, beta): [[-4.0, 4.0], [-1.0, 0.0]] eigenvalues [-2.+0.j -2.+0.j]
2 % settling time 2.92 s


## 5. Breaking one condition at a time

In [8]:
hit = lambda q: q[1] > 3.5 + 0.005 or np.hypot(d.GOAL[0] - q[0], d.GOAL[1] - q[1]) < d.STOP_RHO
for k in [(1, 5, 4), (1, 0.5, -4)]:
    r = d.run(lambda q: d.pose_law(q, k=k), t_max=30, stop=hit)
    q = (r['x'][-1], r['y'][-1], r['th'][-1])
    print(k, f'ends at {r["t"][-1]:.2f} s at ({q[0]:.3f}, {q[1]:.3f}), {d.GOAL[0] - q[0]:.3f} m left of the dock; rho {d.polar(q)[0]:.3f} m')

(1, 5, 4) ends at 4.70 s at (2.814, 3.524), 0.786 m left of the dock; rho 0.787 m
(1, 0.5, -4) ends at 30.01 s at (3.132, 1.688), 0.468 m left of the dock; rho 1.871 m


## 6. Eight starts, and heading gains slower than the distance

In [9]:
for ang in np.arange(200, 341, 20):
    A_ = np.radians(ang); q0 = (d.GOAL[0] + 2 * np.cos(A_), d.GOAL[1] + 2 * np.sin(A_), np.radians(90))
    r = d.run(d.pose_law, q0=q0, t_max=30)
    print(f'start at {ang} deg ({q0[0]:.2f}, {q0[1]:.2f}): alpha0 {deg(d.polar(q0)[1]):.1f} deg; arrives {r["t"][-1]:.2f} s facing {deg(r["th"][-1]):.3f} deg')
ths = []
for ang in np.arange(200, 341, 20):
    A_ = np.radians(ang); q0 = (d.GOAL[0] + 2 * np.cos(A_), d.GOAL[1] + 2 * np.sin(A_), np.radians(90))
    r = d.run(lambda q: d.pose_law(q, k=(1, 3, -1)), q0=q0, t_max=30)
    ths.append(deg(r['th'][-1]))
print(f'with k = (1, 3, -1) (natural frequency 1 rad/s, as fast as rho): arrival headings {min(ths):.1f} to {max(ths):.1f} deg')

start at 200 deg (1.72, 2.82): alpha0 -70.0 deg; arrives 7.80 s facing 89.955 deg
start at 220 deg (2.07, 2.21): alpha0 -50.0 deg; arrives 7.15 s facing 89.960 deg
start at 240 deg (2.60, 1.77): alpha0 -30.0 deg; arrives 6.70 s facing 89.972 deg
start at 260 deg (3.25, 1.53): alpha0 -10.0 deg; arrives 6.45 s facing 89.991 deg
start at 280 deg (3.95, 1.53): alpha0 10.0 deg; arrives 6.45 s facing 90.009 deg
start at 300 deg (4.60, 1.77): alpha0 30.0 deg; arrives 6.70 s facing 90.028 deg
start at 320 deg (5.13, 2.21): alpha0 50.0 deg; arrives 7.15 s facing 90.040 deg
start at 340 deg (5.48, 2.82): alpha0 70.0 deg; arrives 7.80 s facing 90.045 deg
with k = (1, 3, -1) (natural frequency 1 rad/s, as fast as rho): arrival headings 84.7 to 95.3 deg


## 7. Nav2's graceful controller (SmoothControlLaw, plugin defaults)

In [10]:
G = d.run(d.graceful_law)
print(f'first command v {d.graceful_law(d.START)[0]:.4f} m/s, w {d.graceful_law(d.START)[1]:.4f} rad/s')
print(f'arrives at {G["t"][-1]:.2f} s facing {deg(G["th"][-1]):.3f} deg; top speed {G["v"].max():.3f} m/s; top turn rate {np.abs(G["w"]).max():.3f} rad/s')
GP = np.array([d.polar(q)[0] for q in zip(G['x'], G['y'], G['th'])])
k = int(np.argmax(GP <= 1.5)); print(f'within 1.5 m of the dock at {G["t"][k]:.2f} s; speed then {G["v"][k]:.3f} m/s')
kslow = int(np.argmax((np.arange(len(G['v'])) > k) & (np.r_[0.0, np.diff(G['v'])] < -1e-3)))
print(f'the slow-down takes over at {G["t"][kslow]:.2f} s, {GP[kslow]:.4f} m from the dock; speed just before {G["v"][kslow - 1]:.3f} m/s')
print(f'last commanded speed {G["v"][-2]:.3f} m/s (v_linear_min)')

first command v 0.4946 m/s, w 0.0814 rad/s
arrives at 11.15 s facing 89.950 deg; top speed 0.495 m/s; top turn rate 0.199 rad/s
within 1.5 m of the dock at 3.35 s; speed then 0.468 m/s
the slow-down takes over at 3.60 s, 1.3848 m from the dock; speed just before 0.466 m/s
last commanded speed 0.100 m/s (v_linear_min)
